---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [233]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [234]:
options(verbose = FALSE)
options(warn = -1)


In [235]:
library(here)
source(here("data-cleaning", "r_scripts", "libraries.R"))
# source(here("data-cleaning", "r_scripts", "everything.R"))


In [236]:
options(warn = 1)


## Set Parameters

### Year to Load, Version, and Parameters

In [237]:
source(here("data-cleaning", "r_scripts", "parameters.R"))


## Source Data Formats

In [238]:
source(here("data-cleaning", "r_scripts", "data-formats.R"))
source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [239]:
source(here("data-cleaning", "r_scripts", "general-functions.R"))
# source(here("data-cleaning", "r_scripts", "clean-data-mini-functions.R"))
# source(here("data-cleaning", "r_scripts", "profvis.R"))
source(here("data-cleaning", "r_scripts", "main-functions.R"))
source(here("data-cleaning", "r_scripts", "icd-functions.R"))
source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [240]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]
# head(rvs_icd9)

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))
# head(acr_rvs)

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")
# head(tdrg_icd10)

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [241]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."


In [242]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Chunking

In [243]:
# if (to_chunk) {
#   tic("Total execution time:")
#   num_cores <- max(1, availableCores() - 1)

#   result <- parallelize_and_summarize(
#     dt, num_cores,
#     to_view_checks = FALSE, global_seed,
#     rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx
#   )

#   dt <- result$dt
#   consolidated_summary <- result$consolidated_summary
#   aggregate_statistics <- result$aggregate_statistics

#   # Print the consolidated summary
#   print(consolidated_summary)

#   # Print aggregate summary statistics
#   print_aggregate_summary_stats(aggregate_statistics, rows_to_show)
# } else if (!to_chunk) {
#   warning("Error: to_chunk must be TRUE; not chunking is deprecated.")
#   print("Setting to_chunk to TRUE and continuing.")
#   to_chunk <- TRUE
#   tic("Total execution time:")
#   num_cores <- max(1, availableCores() - 1)

#   result <- parallelize_and_summarize(
#     dt, num_cores,
#     to_view_checks = FALSE, global_seed,
#     rows_to_show, rvs_icd9, tdrg_icd10, acc_pdx
#   )

#   dt <- result$dt
#   consolidated_summary <- result$consolidated_summary
#   aggregate_statistics <- result$aggregate_statistics

#   # Print the consolidated summary
#   print(consolidated_summary)

#   # Print aggregate summary statistics
#   print_aggregate_summary_stats(aggregate_statistics, rows_to_show)
# }


$rename_success
[1] TRUE

$ICD_replacements_1
    old_code new_code count
      <char>   <char> <int>
 1:   J18.92    J1892  1569
 2:    A09.9     A099   849
 3:    N39.0     N390   728
 4:    A97.1     A971   392
 5:    K29.1     K291   277
 6:    I10.1     I101   251
 7:    I10.9     I109   251
 8:   J45.90    J4590   241
 9:    A97.0     A970   234
10:    P36.9     P369   211

$ICD_replacements_2
    old_code new_code count
      <char>   <char> <int>
 1:    I21.9     I219    15
 2:    E86.1     E861     9
 3:    I63.9     I639     6
 4:    I21.0     I210     4
 5:    I21.4     I214     3
 6:    I61.9     I619     3
 7:    I63.4     I634     2
 8:    I63.8     I638     2
 9:    O67.8     O678     2
10:    I61.5     I615     1

$pat_type_unmapped
NULL

$memcat_parent_unmapped
NULL

$memcat_child_unmapped
NULL

$discharge_unmapped
NULL

$discard_rvs_one
      CODE count
    <char> <int>
 1:  77401   267
 2:  77418   133
 3:  90375   126
 4:  77421    25
 5:  77261    14
 6:  76942    

#### Not Chunking

##### Clean Data

In [244]:
if (!to_chunk) {
  if (to_profvis) {
    tic("Total execution time:")
    p <- profvis({
      clean_result <- clean_data(dt)
      dt <- clean_result$data

      summary <- list()
      summary$rename_success <- clean_result$rename_success
      summary$ICD_replacements_1 <- clean_result$ICD_replacements_1
      summary$ICD_replacements_2 <- clean_result$ICD_replacements_2
      summary$pat_type_unmapped <- clean_result$pat_type_unmapped
      summary$memcat_parent_unmapped <- clean_result$memcat_parent_unmapped
      summary$memcat_child_unmapped <- clean_result$memcat_child_unmapped
      summary$discharge_unmapped <- clean_result$discharge_unmapped
      summary$discard_rvs_one <- clean_result$discard_rvs_one
      summary$discard_rvs_two <- clean_result$discard_rvs_two
      summary$empty_strings_replaced_1 <- clean_result$empty_strings_replaced_1
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "clean_data.html"),
      selfcontained = TRUE
    )
  } else {
    tic("Total execution time:")
    clean_result <- clean_data(dt)
    dt <- clean_result$data
  }
}


##### Map Codes

In [245]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      rvs_mapping_result <- map_rvs_icd9(dt$clin_rvs, rvs_icd9)
      dt[, icd9_list := rvs_mapping_result$icd9_list]
      map_then_compare_icd_mappings(
        tdrg_icd10,
        rows_to_show,
        invalid_rows_to_show = rows_to_show
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "map_rvs.html"),
      selfcontained = TRUE
    )
  } else {
    rvs_mapping_result <- map_rvs_icd9(dt$clin_rvs, rvs_icd9)
    dt[, icd9_list := rvs_mapping_result$icd9_list]
    map_then_compare_icd_mappings(
      tdrg_icd10,
      rows_to_show,
      invalid_rows_to_show = rows_to_show
    )
  }
}


##### Find PDx

In [246]:
# Find PDX
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "find_pdx.html"),
      selfcontained = TRUE
    )
  } else {
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


## Export

### Export Intermediate Output

In [247]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


### Export for Batch Grouper

In [248]:
if (to_group) {
  if (to_profvis) {
    p <- profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "export_for_grouper.html"),
      selfcontained = TRUE
    )
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [249]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 6.44 sec elapsed


### Calculate Speed

In [250]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))


Time spent (total) for 25.0k rows:  6.44 sec  (actual)
Time spent (t/row) for 25.0k rows:  0.26 msec (actual)
Time spent (total) for  11.8m rows: 50.57 min  (estimate)


In [251]:
# # List all .R files in the directory
# r_files <- list.files(path = here("data-cleaning", "r_scripts"), pattern = "\\.R$", full.names = TRUE)

# # Output file
# output_file <- "everything.R"

# # Read and concatenate contents
# file_contents <- lapply(r_files, readLines)
# concatenated_content <- unlist(file_contents)
# cat(concatenated_content,
#   file = paste0(here("data-cleaning", "everything", output_file)), sep = "\n"
# )
